<a href="https://colab.research.google.com/github/DeepLabCut/DeepLabCut/blob/main/examples/COLAB/COLAB_Person_reID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Person ReID with DeepLabCut Transformer
![DLC](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1628250004229-KVYD7JJVHYEFDJ32L9VJ/DLClogo2021.jpg?format=1000w)

This notebook demonstrates how to create a project for multi-person pose estimation using the `superanimal_humanbody` model and how to apply the `transformer_reID` utility for unsupervised identity tracking.

## 1. Install DeepLabCut and dependencies
The Colab environment uses CUDA 12 and Python 3.11, so we first install the correct versions of PyTorch, TensorFlow, and DeepLabCut.

In [ ]:
!pip install torch==2.3.1 torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install deeplabcut[tf] -q

## 2. Create a project using the SuperAnimal human model
Here we create a new project initialized with the `superanimal_humanbody_rtmpose_x` weights. Replace `myvideo.mp4` with the path to your own video containing multiple people.

In [ ]:

import deeplabcut
from pathlib import Path

# Path to your multi-person video
video_path = "myvideo.mp4"

# Create the project
config_path, train_cfg = deeplabcut.create_pretrained_project(
    "person-reid",
    "experimenter",
    [video_path],
    model="superanimal_humanbody",
    engine=deeplabcut.Engine.PYTORCH,
    copy_videos=True,
    analyzevideo=False,
)
print("Project created at", config_path)


## 3. Analyze the video
Run pose estimation on the video to obtain detection results.

In [ ]:
deeplabcut.analyze_videos(config_path, [video_path], videotype="mp4")

## 4. Learn identities with `transformer_reID`
The following call mines triplets from the detections, trains a transformer, and stitches the tracklets.

In [ ]:

deeplabcut.transformer_reID(
    config_path,
    [video_path],
    n_tracks=2,  # adjust for number of people
    track_method="ellipse",
    train_epochs=100,
    destfolder=None,
)


## 5. Visualize the results
Create a labeled video to inspect the stitched tracks.

In [ ]:
deeplabcut.create_labeled_video(config_path, [video_path], track_method="transformer")